# Overfitting with High-Degree Polynomials

A **high-degree** poly fits noise on training data but fails on validation.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)


## 1. Synthetic noisy data

In [ ]:
n = 40
x = torch.linspace(-1, 1, n)
y = torch.sin(3 * x) + 0.2 * torch.randn(n)
x_train, x_val = x[:30], x[30:]
y_train, y_val = y[:30], y[30:]

## 2. Fit degree 1 vs degree 9

In [ ]:
def poly_fit(degree, x_tr, y_tr):
    X = torch.stack([x_tr**i for i in range(degree+1)], dim=1)
    w = torch.linalg.lstsq(X, y_tr.unsqueeze(1)).solution.squeeze()
    return w

def predict(w, x):
    deg = len(w) - 1
    return sum(w[i] * x**i for i in range(deg+1))

w1 = poly_fit(1, x_train, y_train)
w9 = poly_fit(9, x_train, y_train)
x_plot = torch.linspace(-1, 1, 200)

## 3. Plot fits

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(x_train, y_train, label='train', color='steelblue', s=40)
ax.scatter(x_val, y_val, label='val', color='coral', s=40)
ax.plot(x_plot, predict(w1, x_plot), 'g-', lw=2, label='degree 1')
ax.plot(x_plot, predict(w9, x_plot), 'r-', lw=2, label='degree 9')
ax.legend(); ax.set_title('Low vs high degree polynomial')
plt.tight_layout(); plt.show()

## 4. Train vs val MSE across degrees

In [ ]:
degrees = range(1, 12)
train_mse, val_mse = [], []
for d in degrees:
    w = poly_fit(d, x_train, y_train)
    train_mse.append(float(((predict(w, x_train) - y_train)**2).mean()))
    val_mse.append(float(((predict(w, x_val) - y_val)**2).mean()))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(list(degrees), train_mse, 'b-o', label='train MSE', lw=2)
ax.plot(list(degrees), val_mse, 'r-o', label='val MSE', lw=2)
ax.set_xlabel('polynomial degree'); ax.legend()
ax.set_title('Validation error rises when overfitting')
plt.tight_layout(); plt.show()

## 5. Gap between train and val error

In [ ]:
gap = np.array(val_mse) - np.array(train_mse)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(list(degrees), gap, color='orange', edgecolor='black')
ax.set_xlabel('degree'); ax.set_ylabel('val_mse - train_mse')
ax.set_title('Generalization gap grows with model complexity')
plt.tight_layout(); plt.show()